In [1]:
import numpy as np

In [ ]:
# Hardcoded Pauli matrices
from re import S


Z = np.array([[1, 0], [0, -1]])
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
S = np.array([[1, 0], [0, 1j]])
Sdag = np.array([[1, 0], [0, -1j]])
CX = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])
I = np.eye(2)
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)

# Hardcoded common states
s0 = np.array([[1], [0]])
s1 = np.array([[0], [1]])
sp = np.array([[1], [1]] / np.sqrt(2))
sm = np.array([[1], [-1]] / np.sqrt(2))



In [3]:
# Operations/helpers
def tp(a, b):
    return np.kron(a, b)

def apply(pauli, state):
    return np.dot(pauli, state)

cellwidth = 6  # Increase width for decimals

def format_number(x):
    # Format complex numbers and real numbers to 2 decimals; handle sign and zero
    if isinstance(x, complex):
        real = round(x.real, 2)
        imag = round(x.imag, 2)
        if real == 0 and imag == 0:
            return " 0.00"
        elif real == 0:
            return f"{imag:+.2f}j"
        elif imag == 0:
            return f"{real: .2f}"
        else:
            return f"{real: .2f}{imag:+.2f}j"
    else:
        return f"{x: .2f}"

def display_multiplication(pauli, state):
    product = apply(pauli, state)
    for i in range(len(pauli)):
        for j in pauli[i]:
            current_cell = format_number(j)
            while len(current_cell) < cellwidth:
                current_cell = " " + current_cell
            print(current_cell, end="")
        print(" " * (cellwidth - 1), end="")
        for j in state[i]:
            current_cell = format_number(j)
            while len(current_cell) < cellwidth:
                current_cell = " " + current_cell
            print(current_cell, end="")
        print(" " * (cellwidth - 1), end="")
        for j in product[i]:
            current_cell = format_number(j)
            while len(current_cell) < cellwidth:
                current_cell = " " + current_cell
            print(current_cell)
    print()
    return product
            
def display_state(state):
    for i in range(len(state)):
        for j in state[i]:
            current_cell = format_number(j)
            while len(current_cell) < cellwidth:
                current_cell = " " + current_cell
            print(current_cell)

def display_pauli_matrix(pauli):
    for i in range(len(pauli)):
        current_row = ""
        for j in pauli[i]:
            current_cell = format_number(j)
            while len(current_cell) < cellwidth:
                current_cell = " " + current_cell
            current_row += current_cell
        print(current_row)

def print_nonzero_indices(state):
    output = ""
    for i in range(len(state)):
        for j in state[i]:
            if j != 0:
                sign = "-" if j < 0 else ""
                output += f"{sign}{i}\t"
    print(output)

def create_stabilizer(sequence):
    outmatrix = sequence[0]
    for gate in sequence[1:]:
        outmatrix = tp(gate, outmatrix)
    return outmatrix


In [4]:
display_pauli_matrix(create_stabilizer([Z, Z, I]))

  1.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00
  0.00 -1.00  0.00  0.00  0.00 -0.00  0.00  0.00
  0.00  0.00 -1.00  0.00  0.00  0.00 -0.00  0.00
  0.00  0.00  0.00  1.00  0.00  0.00  0.00  0.00
  0.00  0.00  0.00  0.00  1.00  0.00  0.00  0.00
  0.00 -0.00  0.00  0.00  0.00 -1.00  0.00  0.00
  0.00  0.00 -0.00  0.00  0.00  0.00 -1.00  0.00
  0.00  0.00  0.00  0.00  0.00  0.00  0.00  1.00


In [5]:
# Create 3 3-qubit GHZ states
state123 = apply(tp(I, CX), apply(tp(CX, I), tp(apply(H, s0), tp(s0, s0))))
state456 = apply(tp(I, CX), apply(tp(CX, I), tp(apply(H, s0), tp(s0, s0))))
state789 = apply(tp(I, CX), apply(tp(CX, I), tp(apply(H, s0), tp(s0, s0))))

# Model phase flips
#state123[7] = state123[7] * -1
#state456[7] = state456[7] * -1
#state789[7] = state789[7] * -1

# Create 6 qubit tensors
state123456 = tp(state123, state456)
state456789 = tp(state456, state789)
# Apply XXXXXX
xxxxxx = tp(X, tp(X, tp(X, tp(X, tp(X, X)))))
state123456x = apply(xxxxxx, state123456)
state456789x = apply(xxxxxx, state456789)
# Print before and after
print_nonzero_indices(state123456)
print_nonzero_indices(state123456x)
print()
print_nonzero_indices(state456789)
print_nonzero_indices(state456789x)

# Model bit flips
state123[0] = 0
state123[-1] = 0
state123[1] = 1 / np.sqrt(2)
state123[-2] = 1 / np.sqrt(2)
display_state(state123)
# Apply ZZ
zzi = tp(Z, tp(Z, I))
statezzi = apply(zzi, state123)
izz = tp(I, tp(Z, Z))
stateizz = apply(izz, state123)
print_nonzero_indices(statezzi)
print_nonzero_indices(stateizz)


0	7	56	63	
0	7	56	63	

0	7	56	63	
0	7	56	63	
  0.00
  0.71
  0.00
  0.00
  0.00
  0.00
  0.71
  0.00
1	6	
-1	-6	


In [6]:
state123p = apply(tp(I, CX), apply(tp(CX, I), tp(apply(H, s0), tp(s0, s0))))
state123m = apply(tp(I, CX), apply(tp(CX, I), tp(apply(H, s1), tp(s0, s0))))
state123p[1] = 1 / np.sqrt(2)
state123p[-2] = 1 / np.sqrt(2)
state123p[0] = 0
state123p[-1] = 0
print_nonzero_indices(state123p)
print_nonzero_indices(state123m)

measuredp = apply(tp(X, tp(X, X)), state123p)
print_nonzero_indices(measuredp)
measuredm = apply(tp(X, tp(X, X)), state123m)
print_nonzero_indices(measuredm)



1	6	
0	-7	
1	6	
-0	7	


In [7]:
state123p = apply(tp(I, CX), apply(tp(CX, I), tp(apply(H, s0), tp(s0, s0))))
state123m = apply(tp(I, CX), apply(tp(CX, I), tp(apply(H, s1), tp(s0, s0))))

print_nonzero_indices(state123p)
print_nonzero_indices(state123m)

state123ph = apply(tp(H, tp(H, H)), state123p)
display_multiplication(tp(H, tp(H, H)), state123p)
state123mh = apply(tp(H, tp(I, I)), state123m)
display_state(state123mh)

measuredp = apply(tp(Z, tp(I, I)), state123ph)
print_nonzero_indices(measuredp)
measuredm = apply(tp(Z, tp(I, I)), state123mh)
print_nonzero_indices(measuredm)



0	7	
0	-7	
  0.35  0.35  0.35  0.35  0.35  0.35  0.35  0.35       0.71       0.50
  0.35 -0.35  0.35 -0.35  0.35 -0.35  0.35 -0.35       0.00       0.00
  0.35  0.35 -0.35 -0.35  0.35  0.35 -0.35 -0.35       0.00       0.00
  0.35 -0.35 -0.35  0.35  0.35 -0.35 -0.35  0.35       0.00       0.50
  0.35  0.35  0.35  0.35 -0.35 -0.35 -0.35 -0.35       0.00       0.00
  0.35 -0.35  0.35 -0.35 -0.35  0.35 -0.35  0.35       0.00       0.50
  0.35  0.35 -0.35 -0.35 -0.35 -0.35  0.35  0.35       0.00       0.50
  0.35 -0.35 -0.35  0.35 -0.35  0.35  0.35 -0.35       0.71       0.00

  0.50
  0.00
  0.00
 -0.50
  0.50
  0.00
  0.00
  0.50
0	3	-5	-6	
0	-3	-4	-7	


In [8]:
state = apply(CX, tp(s0, s0))
state = apply(tp(H, H), state)
display_state(state)

  0.50
  0.50
  0.50
  0.50


In [9]:
# Test bit flip
state = tp(s0, tp(s0, s0))
state[0] = 0
state[6] = 1
z12 = tp(Z, tp(Z, I))
statez12 = apply(z12, state)
print_nonzero_indices(statez12)
z23 = tp(I, tp(Z, Z))
#display_pauli_matrix(z23)
statez23 = apply(z23, state)
print_nonzero_indices(statez23)

6	
-6	


In [13]:
clifford = X @ H
display_pauli_matrix(clifford)

  0.71 -0.71
  0.71  0.71
